# 03 — Motors and Rigid Body Motion

This notebook introduces **motors** — the Projective Geometric Algebra representation of rigid body transformations. A motor combines translation and rotation into a single even-grade element, enabling elegant kinematics computations.

## Learning Objectives

- Understand motors as the PGA representation of rigid motions
- Construct translators (pure translation)
- Construct rotors (pure rotation)
- Compose translators and rotors into motors
- Apply motors using sandwich conjugation
- Visualize trajectory evolution

In [ ]:
# Setup
from amsa import Algebra
import numpy as np
import matplotlib.pyplot as plt

alg = Algebra.pga2d()

## 3.1 What is a Motor?

A **motor** is an even-grade element in PGA that represents a rigid body transformation:

$$M = T \cdot R = \text{translator} \times \text{rotor}$$

Properties:
- **Even grade**: grades 0 (scalar) + 2 (bivector) only
- **Dual representation**: encodes both rotation center and translation direction
- **Sandwich application**: $P' = M P \tilde{M}$ applies the transformation

This is fundamentally simpler than homogeneous transformation matrices!

## 3.2 Translators (Pure Translation)

A translator in PGA2d moves points along a direction. It's constructed from a vector:

$$T = 1 + \frac{1}{2} d \cdot e_0$$

where $d$ is the translation vector. The factor of 1/2 comes from the sandwich product mechanics.

In [ ]:
# Create a translator for (0.5, 0) movement in x-direction
dx, dy = 0.5, 0.0

translator = alg.multivector({
    "e": 1.0,
    "e02": 0.5 * dx,  # Note: uses e02 (y-direction displacement encoded in e2^e0)
})

# Alternative: using e01 for x-translation, e02 for y-translation
translator_x = alg.multivector({"e": 1.0, "e01": 0.5 * dx})

print("Translator:", translator.values)
print("Translator (x-direction):", translator_x.values)

## 3.3 Rotors (Pure Rotation)

Rotors in PGA2d are identical to VGA2d rotors — they live in the e + e12 subspace:

$$R = \cos(\theta/2) - e_{12} \sin(\theta/2)$$

This rotates around the origin in the plane.

In [ ]:
# Create a rotor for 30-degree rotation
theta = np.deg2rad(30)

rotor = alg.multivector({
    "e": np.cos(theta / 2),
    "e12": -np.sin(theta / 2)
}).normalized()

print("Rotor (30°):", rotor.values)

# Apply rotor to a point
point = alg.multivector({"e01": 1.0, "e02": 0.0, "e12": 1.0})  # (1, 0)
rotated = rotor.sandwich(point)

print("\nOriginal point (1, 0):", point.component("e01"), point.component("e02"))
print("Rotated point:", rotated.component("e01"), rotated.component("e02"))

## 3.4 Composing Motors (Translation × Rotation)

A motor combines translation and rotation:

$$M = T \times R$$

The order matters: applying $R$ then $T$ vs $T$ then $R$ gives different results.

In PGA, the convention is: **translate first, then rotate** (from right to left in the product).

In [ ]:
# Create a motor: translate by (0.5, 0), then rotate by 20°
dx, dy = 0.5, 0.0
theta = np.deg2rad(20)

# Translator
translator = alg.multivector({
    "e": 1.0,
    "e01": 0.5 * dx,
    "e02": 0.5 * dy,
})

# Rotor
rotor = alg.multivector({
    "e": np.cos(theta / 2),
    "e12": -np.sin(theta / 2)
}).normalized()

# Motor = translator * rotor
motor = translator * rotor

print("Translator:", translator.values)
print("Rotor:", rotor.values)
print("\nMotor (T × R):", motor.values)

## 3.5 Applying Motors to Points

Like rotors, motors are applied using the sandwich product:

$$P' = M P \tilde{M}$$

This handles both translation and rotation in one step!

In [ ]:
# Apply motor to origin point
origin = alg.multivector({"e01": 0.0, "e02": 0.0, "e12": 1.0})

transformed = motor.sandwich(origin)

print("Original origin (0, 0):")
print("  x =", origin.component("e01"), ", y =", origin.component("e02"))

print("\nAfter motor transform:")
print("  x =", transformed.component("e01"), ", y =", transformed.component("e02"))

## 3.6 Visualizing Motor Trajectory

Let's create a trajectory by repeatedly applying a motor — this simulates a robot moving along a path.

In [ ]:
# Create a motor for small forward + rotation steps
theta_step = np.deg2rad(10)   # 10° per step
forward_step = 0.3

# Translator
T = alg.multivector({
    "e": 1.0,
    "e02": 0.5 * forward_step,
})

# Rotor
R = alg.multivector({
    "e": np.cos(theta_step / 2),
    "e12": -np.sin(theta_step / 2)
}).normalized()

# Motor
M = T * R

# Starting point
point = alg.multivector({"e01": 0.0, "e02": 0.0, "e12": 1.0})

# Compute trajectory
steps = 24  # full circle
trajectory = []

for i in range(steps):
    point = M.sandwich(point)
    x = point.component("e01")
    y = point.component("e02")
    trajectory.append([x, y])

trajectory = np.array(trajectory)

# Visualize
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(trajectory[:, 0], trajectory[:, 1], 'b-', linewidth=1.5, alpha=0.7)
ax.scatter(trajectory[:, 0], trajectory[:, 1], c=range(steps), cmap='viridis', s=30, zorder=5)
ax.scatter(0, 0, c='red', s=100, marker='*', zorder=6, label='Start')

ax.set_xlim(-1, 3)
ax.set_ylim(-1, 3)
ax.set_aspect('equal')
ax.set_title('Motor Trajectory: Forward + Turn', fontsize=12)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 3.7 Motor Inverse

The inverse of a motor (like a rotor) is its reverse:

$$M^{-1} = \tilde{M}$$

This makes backward transformations trivial!

In [ ]:
# Test inverse
M_inv = motor.reverse()

# Original point
point = alg.multivector({"e01": 1.0, "e02": 1.0, "e12": 1.0})

# Forward
forward = motor.sandwich(point)

# Backward
backward = M_inv.sandwich(forward)

print("Original:", point.component("e01"), point.component("e02"))
print("After forward:", forward.component("e01"), forward.component("e02"))
print("After backward:", backward.component("e01"), backward.component("e02"))
print("\nRestored:", np.allclose([point.component("e01"), point.component("e02")], 
                               [backward.component("e01"), backward.component("e02")], atol=1e-10))

## 3.8 Comparison: Motor vs Transformation Matrix

Let's verify that a PGA motor gives the same result as a classical homogeneous transformation matrix.

In [ ]:
# Classical transformation: rotate 30°, translate (1, 0.5)
theta = np.deg2rad(30)
tx, ty = 1.0, 0.5

c, s = np.cos(theta), np.sin(theta)
T_matrix = np.array([
    [c, -s, tx],
    [s, c, ty],
    [0, 0, 1]
])

# Equivalent PGA motor
translator = alg.multivector({"e": 1.0, "e01": 0.5*tx, "e02": 0.5*ty})
rotor = alg.multivector({"e": np.cos(theta/2), "e12": -np.sin(theta/2)}).normalized()
M_pga = translator * rotor

# Test point
p_homogeneous = np.array([2.0, 1.0, 1.0])  # (2, 1)

# Matrix result
p_matrix = T_matrix @ p_homogeneous

# PGA result
p_pga = alg.vector(p_homogeneous)
p_pga_transformed = M_pga.sandwich(p_pga)
p_pga_result = [p_pga_transformed.component("e01"), p_pga_transformed.component("e02")]

print("Matrix result:", p_matrix[:2])
print("PGA result:", p_pga_result)
print("\nMatch:", np.allclose(p_matrix[:2], p_pga_result))

## 3.9 Summary

We covered:

- **Translators**: Pure translation in PGA2d (e + e01/e02 terms)
- **Rotors**: Pure rotation (e + e12 terms)
- **Motors**: Combined translation + rotation = $T \times R$
- **Sandwich application**: $M P \tilde{M}$ applies the transformation
- **Motor inverse**: Simply $\tilde{M}$
- **Equivalence**: Motors match homogeneous matrices exactly

In the next notebook, we'll explore **bulk and weight** — the null basis decomposition essential for PGA normalization.

## Exercises

### ⭐ Easy

**3.1** Create a motor that translates by (2, 0) and rotates by 45°. Apply it to the point (1, 1). What are the resulting coordinates?

In [ ]:
# Your turn: ⭐ Exercise 3.1
# TODO: Create motor and apply to point (1,1)
raise NotImplementedError("Implement exercise 3.1")

### ⭐⭐ Medium

**3.2** Write code to compute a motor that moves from point A to point B with a given rotation angle. Apply it to several points and verify they form the expected transformed shape.

In [ ]:
# Your turn: ⭐⭐ Exercise 3.2
# Create a motor from (0,0) to (2,1) with 30° rotation
def motor_from_to(start, end, angle):
    """Create motor that translates and rotates from start to end."""
    # TODO: Implement
    raise NotImplementedError("Implement motor_from_to function")

M = motor_from_to((0, 0), (2, 1), np.pi/6)
# Test on a square of points
test_points = [(0, 0), (1, 0), (1, 1), (0, 1)]
# TODO: Transform and visualize
raise NotImplementedError("Implement exercise 3.2")

### ⭐⭐⭐ Challenge

**3.3** Create a function that interpolates between two motors: given motors M1 and M2 and parameter t ∈ [0, 1], compute the interpolated motor. This is useful for motion planning. Hint: use the exponential map (log of motors).

In [ ]:
# Your turn: ⭐⭐⭐ Exercise 3.3
def interpolate_motors(M1, M2, t):
    """Interpolate between two motors."""
    # TODO: Use log/exp for smooth interpolation
    raise NotImplementedError("Implement interpolate_motors function")

# Create two motors
T1 = alg.multivector({"e": 1.0, "e01": 0.0})
R1 = alg.multivector({"e": np.cos(np.pi/8), "e12": -np.sin(np.pi/8)}).normalized()
M1 = T1 * R1

T2 = alg.multivector({"e": 1.0, "e01": 2.0})
R2 = alg.multivector({"e": np.cos(np.pi/4), "e12": -np.sin(np.pi/4)}).normalized()
M2 = T2 * R2

# Interpolate at t=0.5
M_mid = interpolate_motors(M1, M2, 0.5)
print("Mid-motor:", M_mid.values)

## Attribution

This notebook draws on:

- **SIGGRAPH 2019 Course Notes** — Charles G. Gunn
  https://arxiv.org/abs/2002.04509
- **Projective Geometric Algebra** — Charles G. Gunn
  https://arxiv.org/abs/1901.05873
- **Geometric Algebra for Computer Graphics** — John Vince
  https://link.springer.com/book/10.1007/978-1-84628-997-2
- **PGABLE Tutorial** — Leger and Mann
  https://cs.uwaterloo.ca/~smann/PGABLE/PGAtutorial.pdf